### Numerical tidal current prediction direction and velocity

In [1]:
%useLatestDescriptors
%use dataframe
%use datetime

In [179]:
USE {
    dependencies("org.xerial:sqlite-jdbc:3.51.1.0")
}

In [202]:
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import kotlin.time.Clock
import kotlin.time.ExperimentalTime

@OptIn(ExperimentalTime::class, FormatStringsInDatetimeFormats::class)
var now = Clock.System.now()
@OptIn(ExperimentalTime::class, FormatStringsInDatetimeFormats::class)
val preTime = now.minus(5, DateTimeUnit.MINUTE).toLocalDateTime(TimeZone.of("Asia/Seoul")).format(LocalDateTime.Format { byUnicodePattern("yyyy-MM-dd HH:mm") })

val url = "jdbc:sqlite:/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite"

val dbConfig = DbConnectionConfig(url)
val tableName = "TidalCurrentInfoKHOA"
val query = "SELECT * FROM $tableName WHERE sch_time >= '${preTime}' ORDER BY sch_time ASC"

val df = DataFrame.readSqlQuery(dbConfig, query)
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
sch_time,String,11016,12,0,2026-03-22 19:25,918,null,null,2026-03-22 19:25,2026-03-22 19:35,2026-03-22 19:50,2026-03-22 20:05,2026-03-22 20:20
pre_lon,Double,11016,918,0,124.000080,12,126.450835,1.723807,124.000080,125.004770,126.057135,127.933760,129.861370
pre_lat,Double,11016,918,0,33.153920,12,34.307681,1.501861,32.003590,33.079380,34.094000,35.381090,37.789130
current_dir,Double,11016,349,0,226.000000,118,208.589325,62.687970,0.000000,166.000000,210.000000,245.000000,360.000000
current_speed,Double,11016,157,0,16.000000,240,50.065087,28.342671,1.000000,26.000000,48.000000,71.000000,161.000000


In [ ]:
fun Double.round(decimals: Int): Double {
    var multiplier = 1.0
    repeat(decimals) { multiplier *= 10 }
    return kotlin.math.round(this * multiplier) / multiplier
}


In [208]:
data class TidalCurrentInfo(
    val sch_time :String,
    val pre_lon:Double,
    val pre_lat:Double,
    val current_dir:Double,
    val current_speed :Double
)

data class TidalCurrentData(
    val schTime: String,
    val currentDir: Double,
    val currentSpeed: Double,
    var predict_lat:Double,
    var predict_lon:Double
)

val Double.toRadians get() = this * (PI / 180.0)

fun transformTidalList(tidalList: List<TidalCurrentInfo>): Map<Pair<Double, Double>, List<TidalCurrentData>> {

    // 2. 그룹화 및 변환 실행
    return tidalList.groupBy(
        // Key: 위경도 쌍 (pre_lon, pre_lat)
        keySelector = {  it.pre_lat to it.pre_lon },
        // Value 변환: 각 항목에서 필요한 값 추출 및 U, V 계산
        valueTransform = { item ->

            TidalCurrentData(
                schTime = item.sch_time,
                currentDir = item.current_dir,
                currentSpeed = item.current_speed,
                predict_lat = item.pre_lat,
                predict_lon = item.pre_lon
            )
        }
    )


}



In [214]:
val tidalList = df.toListOf<TidalCurrentInfo>()
val result = transformTidalList(tidalList)


In [220]:

fun updatePredictCoordinates2(
    result: Map<Pair<Double, Double>, List<TidalCurrentData>>,
    timeIntervalSeconds: Double = 300.0 // 5분 간격 기준
) {
    val earthRadius = 6371000.0 // m

    result.forEach { (coords, dataList) ->


        dataList.forEachIndexed { index, data ->

            if(index > 0 ){
                val speedMps = dataList[index-1].currentSpeed * 0.01
                val distance = speedMps * timeIntervalSeconds * index

                val lat1 = Math.toRadians(dataList[index-1].predict_lat)
                val lon1 = Math.toRadians(dataList[index-1].predict_lon)

                val brng = Math.toRadians(dataList[index-1].currentDir)


                val lat2 = asin(
                    sin(lat1) * cos(distance / earthRadius) +
                            cos(lat1) * sin(distance / earthRadius) * cos(brng)
                )

                val lon2 = lon1 + atan2(
                    sin(brng) * sin(distance / earthRadius) * cos(lat1),
                    cos(distance / earthRadius) - sin(lat1) * sin(lat2)
                )

                // 6. 결과 세팅 (소수점 6자리까지 반올림 - 위경도는 정밀도가 중요하므로 6자리 권장)
                data.predict_lat = (Math.toDegrees(lat2) * 1000000.0).roundToLong() / 1000000.0
                data.predict_lon = (Math.toDegrees(lon2) * 1000000.0).roundToLong() / 1000000.0


            }

        }
    }
}

In [195]:

fun updatePredictCoordinates(
    result: Map<Pair<Double, Double>, List<TidalCurrentData>>,
    timeIntervalSeconds: Double = 300.0 // 5분 간격 기준
) {
    val earthRadius = 6371000.0 // m

    result.forEach { (coords, dataList) ->


        dataList.forEachIndexed { index, data ->

            val speedMps = data.currentSpeed * 0.01
            val distance = speedMps * timeIntervalSeconds * (index+1)

            // 3. 라디안 변환
            val lat1 = Math.toRadians(coords.first)
            val lon1 = Math.toRadians(coords.second)

            val brng = Math.toRadians(data.currentDir)


            val lat2 = asin(
                sin(lat1) * cos(distance / earthRadius) +
                        cos(lat1) * sin(distance / earthRadius) * cos(brng)
            )

            val lon2 = lon1 + atan2(
                sin(brng) * sin(distance / earthRadius) * cos(lat1),
                cos(distance / earthRadius) - sin(lat1) * sin(lat2)
            )

            // 6. 결과 세팅 (소수점 6자리까지 반올림 - 위경도는 정밀도가 중요하므로 6자리 권장)
            data.predict_lat = (Math.toDegrees(lat2) * 1000000.0).roundToLong() / 1000000.0
            data.predict_lon = (Math.toDegrees(lon2) * 1000000.0).roundToLong() / 1000000.0
        }
    }
}

In [221]:
updatePredictCoordinates2(result)

In [222]:
val keys = result.map{it}.joinToString(
    separator = ",",
    prefix = "[",
    postfix = "]"
) { it ->

    "{ lat: ${it.key.first}, lng: ${it.key.second} }"
}


val values = result.map{it}.joinToString(
    separator = ",",
    prefix = "[",
    postfix = "]"
){ it ->
    val data = it.value.joinToString (
        separator = ",",
        prefix = "[",
        postfix = "]"
    ){ (schTime, currentDir, currentSpeed, predict_lat, predict_lon) ->
        "{lat:${predict_lat}, lng:${predict_lon}, speed:${currentSpeed}}"
    }
    data
}


In [223]:
File("/Users/unchil/AndroidStudioProjects/OceanWaterInfo/composeApp/src/jvmMain/resources/keys.js").writeText(keys)

In [224]:
File("/Users/unchil/AndroidStudioProjects/OceanWaterInfo/composeApp/src/jvmMain/resources/values.js").writeText(values)